# LizyML Tutorial: Regression with LightGBM + Tuning

This notebook demonstrates hyperparameter tuning with LizyML:

1. Data preparation (Diamonds dataset with categorical columns)
2. Config with search space
3. Model tuning — Optuna-backed hyperparameter search
4. Tuning table — explore all trial results
5. Model fit — best params auto-applied
6. Evaluate — IS/OOS metrics + learning curve

**Dataset**: Diamonds (~54,000 rows)
**Target**: `price` (USD)
**Categorical features**: `cut`, `color`, `clarity`

## 1. Setup

In [ ]:
import pandas as pd

from lizyml import Model

## 2. Data Preparation

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
df = pd.read_csv(url)
print(f"Shape: {df.shape}")
df.head()

## 3. Config

All 14 LightGBM parameters are explicitly defined in the search space across three categories:

- **model** (9): `objective`, `n_estimators`, `learning_rate`, `max_depth`, `feature_fraction`, `bagging_fraction`, `bagging_freq`, `lambda_l2`, `max_bin`
- **smart** (3): `num_leaves_ratio`, `min_data_in_leaf_ratio`, `min_data_in_bin_ratio` (data-size-relative, resolved per trial)
- **training** (2): `early_stopping_rounds`, `validation_ratio`

Fixed params (`auto_num_leaves=True`) are set at the model config level.

In [ ]:
config = {
    "config_version": 1,
    "task": "regression",
    # --- data ---
    "data": {"target": "price"},
    # --- features ---
    "features": {
        "exclude": [],
        "auto_categorical": True,
        "categorical": [],  # auto_categorical detects string/category columns
    },
    # --- split ---
    "split": {
        "method": "kfold",  # regression default
        "n_splits": 5,
        "random_state": 42,
        "shuffle": True,
    },
    # --- model ---
    "model": {
        "name": "lgbm",
        "params": {},  # tuning will override per trial
        "auto_num_leaves": True,
        "num_leaves_ratio": 1.0,
        "min_data_in_leaf_ratio": 0.01,
        "min_data_in_bin_ratio": 0.01,
        "feature_weights": None,
        "balanced": False,
    },
    # --- training ---
    "training": {
        "seed": 42,
        "early_stopping": {
            "enabled": True,
            "rounds": 150,
            "validation_ratio": 0.1,
        },
    },
    # --- tuning ---
    "tuning": {
        "optuna": {
            "params": {"n_trials": 2, "direction": "minimize"},
            "space": {
                # --- model category (9) ---
                "objective": {
                    "type": "categorical",
                    "choices": ["huber", "mae"],
                    "category": "model",
                },
                "n_estimators": {
                    "type": "int",
                    "low": 600,
                    "high": 2500,
                    "category": "model",
                },
                "learning_rate": {
                    "type": "float",
                    "low": 0.0001,
                    "high": 0.1,
                    "log": True,
                    "category": "model",
                },
                "max_depth": {
                    "type": "int",
                    "low": 3,
                    "high": 12,
                    "category": "model",
                },
                "feature_fraction": {
                    "type": "float",
                    "low": 0.5,
                    "high": 1.0,
                    "category": "model",
                },
                "bagging_fraction": {
                    "type": "float",
                    "low": 0.5,
                    "high": 1.0,
                    "category": "model",
                },
                "bagging_freq": {
                    "type": "int",
                    "low": 1,
                    "high": 20,
                    "category": "model",
                },
                "lambda_l2": {
                    "type": "float",
                    "low": 1e-8,
                    "high": 10.0,
                    "log": True,
                    "category": "model",
                },
                "max_bin": {
                    "type": "int",
                    "low": 63,
                    "high": 1023,
                    "category": "model",
                },
                # --- smart category (3) ---
                "num_leaves_ratio": {
                    "type": "float",
                    "low": 0.5,
                    "high": 1.0,
                    "category": "smart",
                },
                "min_data_in_leaf_ratio": {
                    "type": "float",
                    "low": 0.01,
                    "high": 0.2,
                    "category": "smart",
                },
                "min_data_in_bin_ratio": {
                    "type": "float",
                    "low": 0.001,
                    "high": 0.05,
                    "category": "smart",
                },
                # --- training category (2) ---
                "early_stopping_rounds": {
                    "type": "int",
                    "low": 40,
                    "high": 240,
                    "category": "training",
                },
                "validation_ratio": {
                    "type": "float",
                    "low": 0.1,
                    "high": 0.3,
                    "category": "training",
                },
            },
        }
    },
    # --- evaluation ---
    "evaluation": {"metrics": ["mae", "rmse", "r2"]},
}

print("Config keys:", list(config.keys()))
print(f"Search space: {len(config['tuning']['optuna']['space'])} dimensions")

## 4. Tune

In [ ]:
model = Model(config)
result = model.tune(data=df)

print(f"Best score (mae): {result.best_score:.4f}")
print(f"Best params: {result.best_params}")

### 4.1 Tuning Table

`model.tuning_table()` returns a DataFrame with all trial results — trial number, OOF metric score, and each searched parameter value.

In [ ]:
model.tuning_table()

### 4.2 Tuning Plot

`model.tuning_plot()` visualizes trial scores and best-score progression.

In [ ]:
fig = model.tuning_plot()
fig.show()

## 5. Fit

`model.fit()` automatically uses the best params from tuning.

In [ ]:
model.fit(data=df)
print("Fit complete.")

# model.fit_result property provides read-only access
print(f"OOF shape: {model.fit_result.oof_pred.shape}")
print(f"Number of folds: {len(model.fit_result.models)}")

### 5.1 LightGBM Parameters

Config smart params and resolved booster params.

In [ ]:
model.params_table()

In [ ]:
# Feature weights (resolved) — only displayed when feature_weights is configured
booster = model.fit_result.models[0]
fw = booster.params.get("feature_weights")
if fw is not None:
    feature_names = model.fit_result.feature_names
    print("Resolved feature_weights (Fold 0):")
    for name, w in zip(feature_names, fw, strict=False):
        print(f"  {name}: {w}")
else:
    print("feature_weights: not configured (all features equally weighted)")

## 6. Evaluate

In [ ]:
model.evaluate_table()

In [ ]:
model.plot_learning_curve()